# 06 — Hybrid ALS + Popularity Recommender

This notebook uses the best tuned ALS configuration from notebook 05 and builds a hybrid recommender that blends:

1. ALS personalized scores
2. Global popularity scores
3. Recent popularity scores
4. Recency-weighted popularity scores

The goal is to beat the global-popularity baseline on Recall@20 while keeping ALS-style coverage/diversity.

In [1]:
import os
import sys
import json
import math
import time
import gc
import platform
import subprocess
from pathlib import Path
from typing import Dict, Iterable, List, Tuple, Optional

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, save_npz

from IPython.display import display

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 200)

# Helps implicit ALS avoid slow nested OpenBLAS thread pools in many Kaggle runtimes.
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

print("Python:", platform.python_version())
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

Python: 3.12.12
Pandas: 2.3.3
NumPy: 2.0.2


## 1. Install / import ALS dependency

In [2]:
try:
    from implicit.als import AlternatingLeastSquares
    import implicit
    print("implicit:", implicit.__version__)
except Exception:
    print("implicit is not installed. Installing now...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "implicit", "-q"])
    from implicit.als import AlternatingLeastSquares
    import implicit
    print("implicit:", implicit.__version__)

implicit is not installed. Installing now...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 72.2 MB/s eta 0:00:00
implicit: 0.7.3


## 2. Environment and path detection

In [3]:
def detect_environment() -> str:
    if (
        os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None
        or Path("/kaggle/input").exists()
        or Path("/kaggle/working").exists()
    ):
        return "kaggle"
    if os.environ.get("COLAB_RELEASE_TAG") is not None:
        return "colab"
    return "local"


ENVIRONMENT = detect_environment()
IN_KAGGLE = ENVIRONMENT == "kaggle"
IN_COLAB = ENVIRONMENT == "colab"

print("Detected environment:", ENVIRONMENT)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Skipping Google Drive mount because this is not real Colab.")

Detected environment: kaggle
Skipping Google Drive mount because this is not real Colab.


In [4]:
PROJECT_NAME = "hm-recommender"


def find_file(filename: str, search_roots: List[Path], required: bool = True) -> Optional[Path]:
    """Find a file by name under likely Kaggle/Colab/local roots."""
    checked_roots = []

    for root in search_roots:
        if root is None:
            continue
        root = Path(root)
        checked_roots.append(str(root))
        if not root.exists():
            continue

        direct_matches = [
            root / filename,
            root / PROJECT_NAME / filename,
            root / PROJECT_NAME / "data" / "processed" / filename,
            root / PROJECT_NAME / "artifacts" / "popularity_baseline" / filename,
            root / PROJECT_NAME / "artifacts" / "als_hyperparameter_tuning" / filename,
            root / PROJECT_NAME / "artifacts" / "hybrid_als_popularity" / filename,
        ]
        for candidate in direct_matches:
            if candidate.exists():
                return candidate

        matches = sorted(root.rglob(filename))
        if matches:
            return matches[0]

    if required:
        raise FileNotFoundError(f"Could not find {filename}. Checked roots: {checked_roots}")
    return None


if IN_KAGGLE:
    PROJECT_DIR = Path("/kaggle/working") / PROJECT_NAME
    SEARCH_ROOTS = [Path("/kaggle/working"), Path("/kaggle/input"), Path.cwd()]
elif IN_COLAB:
    PROJECT_DIR = Path("/content/drive/MyDrive") / PROJECT_NAME
    SEARCH_ROOTS = [PROJECT_DIR, Path("/content"), Path.cwd()]
else:
    PROJECT_DIR = Path.cwd()
    SEARCH_ROOTS = [PROJECT_DIR, Path.cwd(), Path("/mnt/data")]

ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
HYBRID_ARTIFACTS_DIR = ARTIFACTS_DIR / "hybrid_als_popularity"
HYBRID_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

PROCESSED_DATA_DIR = find_file("train_interactions_aggregated.parquet", SEARCH_ROOTS).parent
BASELINE_ARTIFACTS_DIR = find_file("global_popularity_scores.parquet", SEARCH_ROOTS).parent
BEST_TUNING_CONFIG_PATH = find_file("best_als_hyperparameters.json", SEARCH_ROOTS, required=False)
TUNING_COMPARISON_PATH = find_file("best_tuned_als_vs_popularity_comparison.csv", SEARCH_ROOTS, required=False)

print("PROJECT_DIR:", PROJECT_DIR)
print("PROCESSED_DATA_DIR:", PROCESSED_DATA_DIR)
print("BASELINE_ARTIFACTS_DIR:", BASELINE_ARTIFACTS_DIR)
print("BEST_TUNING_CONFIG_PATH:", BEST_TUNING_CONFIG_PATH)
print("HYBRID_ARTIFACTS_DIR:", HYBRID_ARTIFACTS_DIR)

PROJECT_DIR: /kaggle/working/hm-recommender
PROCESSED_DATA_DIR: /kaggle/input/datasets/albaky7/hm-recommender/hm-recommender/data/processed
BASELINE_ARTIFACTS_DIR: /kaggle/input/datasets/albaky7/hm-popularity-baseline/hm-recommender/artifacts/popularity_baseline
BEST_TUNING_CONFIG_PATH: None
HYBRID_ARTIFACTS_DIR: /kaggle/working/hm-recommender/artifacts/hybrid_als_popularity


In [5]:
required_processed_files = [
    "train_interactions.parquet",
    "val_interactions.parquet",
    "test_interactions.parquet",
    "train_interactions_aggregated.parquet",
    "articles_processed.parquet",
    "article_id_map.parquet",
    "customer_id_map.parquet",
    "sample_submission_processed.parquet",
]

required_baseline_files = [
    "global_popularity_scores.parquet",
    "recency_weighted_popularity_scores.parquet",
    "popularity_baseline_metrics.csv",
]

missing = []
for filename in required_processed_files:
    path = PROCESSED_DATA_DIR / filename
    if path.exists():
        print(f"Found processed file {filename}: {path.stat().st_size / (1024 ** 2):.2f} MB")
    else:
        missing.append(str(path))

for filename in required_baseline_files:
    path = BASELINE_ARTIFACTS_DIR / filename
    if path.exists():
        print(f"Found baseline artifact {filename}: {path.stat().st_size / (1024 ** 2):.2f} MB")
    else:
        missing.append(str(path))

if missing:
    raise FileNotFoundError("Missing required inputs:\n" + "\n".join(missing))

print("All required files found.")

Found processed file train_interactions.parquet: 134.36 MB
Found processed file val_interactions.parquet: 28.11 MB
Found processed file test_interactions.parquet: 28.72 MB
Found processed file train_interactions_aggregated.parquet: 129.63 MB
Found processed file articles_processed.parquet: 6.66 MB
Found processed file article_id_map.parquet: 1.20 MB
Found processed file customer_id_map.parquet: 84.84 MB
Found processed file sample_submission_processed.parquet: 84.86 MB
Found baseline artifact global_popularity_scores.parquet: 2.27 MB
Found baseline artifact recency_weighted_popularity_scores.parquet: 2.87 MB
Found baseline artifact popularity_baseline_metrics.csv: 0.00 MB
All required files found.


## 3. Configuration

In [6]:
TOP_K_VALUES = [12, 20]
PRIMARY_K = 20
RANDOM_SEED = 42

# Use 50k users to tune hybrid weights quickly. Set to None only if you have enough runtime.
HYBRID_TUNING_MAX_EVAL_USERS = 50_000

# After tuning weights, evaluate the best hybrid on the full validation and test warm-start users.
RUN_FULL_BEST_HYBRID_EVALUATION = True
FINAL_TOP_N_HYBRIDS = 1
FINAL_MAX_EVAL_USERS = None

# Candidate pool size. Larger values may improve hybrid quality but increase runtime.
ALS_CANDIDATES = 100
POPULARITY_CANDIDATES = 200
RECOMMEND_BATCH_SIZE = 4096

N_DEMO_USERS = 1000
GENERATE_FULL_SAMPLE_SUBMISSION = False

# Fallback best config from the executed 05 notebook result.
FALLBACK_BEST_ALS_CONFIG = {
    "model_name": "als_f64_reg0p1_it15_alpha20p0",
    "stage": "main_grid",
    "factors": 64,
    "alpha": 20.0,
    "regularization": 0.1,
    "iterations": 15,
}

# Hybrid weights to test. These are intentionally not too many, so runtime stays reasonable.
HYBRID_WEIGHT_CONFIGS = [
    {"hybrid_name": "hybrid_als80_global20", "als": 0.80, "global": 0.20, "recent90": 0.00, "recency": 0.00},
    {"hybrid_name": "hybrid_als70_global30", "als": 0.70, "global": 0.30, "recent90": 0.00, "recency": 0.00},
    {"hybrid_name": "hybrid_als60_global40", "als": 0.60, "global": 0.40, "recent90": 0.00, "recency": 0.00},
    {"hybrid_name": "hybrid_als50_global50", "als": 0.50, "global": 0.50, "recent90": 0.00, "recency": 0.00},
    {"hybrid_name": "hybrid_als70_global20_recent10", "als": 0.70, "global": 0.20, "recent90": 0.10, "recency": 0.00},
    {"hybrid_name": "hybrid_als60_global25_recent15", "als": 0.60, "global": 0.25, "recent90": 0.15, "recency": 0.00},
    {"hybrid_name": "hybrid_als50_global30_recent20", "als": 0.50, "global": 0.30, "recent90": 0.20, "recency": 0.00},
    {"hybrid_name": "hybrid_als60_global20_recency20", "als": 0.60, "global": 0.20, "recent90": 0.00, "recency": 0.20},
    {"hybrid_name": "hybrid_als50_global25_recent15_recency10", "als": 0.50, "global": 0.25, "recent90": 0.15, "recency": 0.10},
    {"hybrid_name": "hybrid_als40_global40_recent20", "als": 0.40, "global": 0.40, "recent90": 0.20, "recency": 0.00},
]

for cfg in HYBRID_WEIGHT_CONFIGS:
    weight_sum = cfg["als"] + cfg["global"] + cfg["recent90"] + cfg["recency"]
    if abs(weight_sum - 1.0) > 1e-6:
        raise ValueError(f"Hybrid weights must sum to 1. Bad config: {cfg}")

np.random.seed(RANDOM_SEED)

print("Number of hybrid configs:", len(HYBRID_WEIGHT_CONFIGS))
print("HYBRID_TUNING_MAX_EVAL_USERS:", HYBRID_TUNING_MAX_EVAL_USERS)
print("RUN_FULL_BEST_HYBRID_EVALUATION:", RUN_FULL_BEST_HYBRID_EVALUATION)

Number of hybrid configs: 10
HYBRID_TUNING_MAX_EVAL_USERS: 50000
RUN_FULL_BEST_HYBRID_EVALUATION: True


## 4. Utility functions

In [7]:
def memory_usage_mb(df: pd.DataFrame) -> float:
    return df.memory_usage(deep=True).sum() / (1024 ** 2)


def print_df_info(df: pd.DataFrame, name: str, show_head: bool = True) -> None:
    print()
    print(name)
    print("-" * 100)
    print("Shape:", df.shape)
    print(f"Memory usage: {memory_usage_mb(df):.2f} MB")
    if show_head:
        display(df.head())


def save_json(obj: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=str)
    print(f"Saved: {path}")


def save_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print(f"Saved: {path} | {path.stat().st_size / (1024 ** 2):.2f} MB")


def save_parquet(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=False, compression="snappy")
    print(f"Saved: {path} | {path.stat().st_size / (1024 ** 2):.2f} MB")

In [8]:
def apk(recommended: List[int], actual: set, k: int) -> float:
    if not actual:
        return 0.0

    score = 0.0
    hits = 0
    seen_recommended = set()

    for rank, item in enumerate(recommended[:k], start=1):
        if item in seen_recommended:
            continue
        seen_recommended.add(item)
        if item in actual:
            hits += 1
            score += hits / rank

    return score / min(len(actual), k)


def ndcgk(recommended: List[int], actual: set, k: int) -> float:
    if not actual:
        return 0.0

    dcg = 0.0
    for rank, item in enumerate(recommended[:k], start=1):
        if item in actual:
            dcg += 1.0 / math.log2(rank + 1)

    ideal_hits = min(len(actual), k)
    idcg = sum(1.0 / math.log2(rank + 1) for rank in range(1, ideal_hits + 1))
    return dcg / idcg if idcg > 0 else 0.0


def precisionk(recommended: List[int], actual: set, k: int) -> float:
    if k <= 0:
        return 0.0
    hits = sum(1 for item in recommended[:k] if item in actual)
    return hits / k


def recallk(recommended: List[int], actual: set, k: int) -> float:
    if not actual:
        return 0.0
    hits = sum(1 for item in recommended[:k] if item in actual)
    return hits / len(actual)


def mrrk(recommended: List[int], actual: set, k: int) -> float:
    if not actual:
        return 0.0
    for rank, item in enumerate(recommended[:k], start=1):
        if item in actual:
            return 1.0 / rank
    return 0.0


def hitratek(recommended: List[int], actual: set, k: int) -> float:
    return float(any(item in actual for item in recommended[:k]))

In [9]:
def build_user_item_sets(df: pd.DataFrame) -> Dict[int, set]:
    grouped = df.groupby("customer_idx", sort=False)["article_idx"].agg(
        lambda values: set(values.to_numpy(dtype=np.int32))
    )
    return grouped.to_dict()


def sample_eval_users(user_ids: List[int], max_users: Optional[int], seed: int = RANDOM_SEED) -> List[int]:
    if max_users is None or len(user_ids) <= max_users:
        return list(user_ids)
    rng = np.random.default_rng(seed)
    return rng.choice(np.array(user_ids, dtype=np.int32), size=max_users, replace=False).astype(int).tolist()


def recommend_from_ranked_items(
    ranked_items: np.ndarray,
    seen_items: Optional[set],
    k: int,
    n_candidates: int = 5000,
) -> List[int]:
    if seen_items is None or len(seen_items) == 0:
        return ranked_items[:k].astype(int).tolist()

    recommendations = []
    scan_limit = min(len(ranked_items), n_candidates)

    for item in ranked_items[:scan_limit]:
        item_int = int(item)
        if item_int not in seen_items:
            recommendations.append(item_int)
            if len(recommendations) == k:
                break

    if len(recommendations) < k:
        for item in ranked_items[scan_limit:]:
            item_int = int(item)
            if item_int not in seen_items:
                recommendations.append(item_int)
                if len(recommendations) == k:
                    break

    return recommendations


def update_metric_sums(
    recommended: List[int],
    actual: set,
    k_values: List[int],
    sums: Dict[int, Dict[str, float]],
    unique_recommended_items_by_k: Dict[int, set],
) -> None:
    for k in k_values:
        top_k_recommended = recommended[:k]
        unique_recommended_items_by_k[k].update(top_k_recommended)
        sums[k]["precision"] += precisionk(recommended, actual, k)
        sums[k]["recall"] += recallk(recommended, actual, k)
        sums[k]["map"] += apk(recommended, actual, k)
        sums[k]["ndcg"] += ndcgk(recommended, actual, k)
        sums[k]["mrr"] += mrrk(recommended, actual, k)
        sums[k]["hitrate"] += hitratek(recommended, actual, k)

## 5. Load data

In [10]:
train_interactions = pd.read_parquet(
    PROCESSED_DATA_DIR / "train_interactions.parquet",
    columns=["t_dat", "customer_idx", "article_idx", "price", "sales_channel_id", "weight"],
)

val_interactions = pd.read_parquet(
    PROCESSED_DATA_DIR / "val_interactions.parquet",
    columns=["t_dat", "customer_idx", "article_idx", "price", "sales_channel_id", "weight"],
)

test_interactions = pd.read_parquet(
    PROCESSED_DATA_DIR / "test_interactions.parquet",
    columns=["t_dat", "customer_idx", "article_idx", "price", "sales_channel_id", "weight"],
)

train_interactions_aggregated = pd.read_parquet(
    PROCESSED_DATA_DIR / "train_interactions_aggregated.parquet",
    columns=["customer_idx", "article_idx", "purchase_count", "implicit_weight"],
)

article_id_map = pd.read_parquet(PROCESSED_DATA_DIR / "article_id_map.parquet")
customer_id_map = pd.read_parquet(PROCESSED_DATA_DIR / "customer_id_map.parquet")
articles = pd.read_parquet(PROCESSED_DATA_DIR / "articles_processed.parquet")
sample_submission = pd.read_parquet(PROCESSED_DATA_DIR / "sample_submission_processed.parquet")

for df in [train_interactions, val_interactions, test_interactions]:
    df["t_dat"] = pd.to_datetime(df["t_dat"])
    df["customer_idx"] = df["customer_idx"].astype("int32")
    df["article_idx"] = df["article_idx"].astype("int32")

train_interactions_aggregated["customer_idx"] = train_interactions_aggregated["customer_idx"].astype("int32")
train_interactions_aggregated["article_idx"] = train_interactions_aggregated["article_idx"].astype("int32")
train_interactions_aggregated["implicit_weight"] = train_interactions_aggregated["implicit_weight"].astype("float32")

print_df_info(train_interactions, "Train interactions")
print_df_info(train_interactions_aggregated, "Train aggregated interactions")
print_df_info(val_interactions, "Validation interactions")
print_df_info(test_interactions, "Test interactions")


Train interactions
----------------------------------------------------------------------------------------------------
Shape: (22235277, 6)
Memory usage: 530.13 MB


,t_dat,customer_idx,article_idx,price,sales_channel_id,weight
0,2018-09-20,2,40179,0.050831,2,1.0
1,2018-09-20,7,46302,0.016932,2,1.0
2,2018-09-20,7,6386,0.020322,2,1.0
3,2018-09-20,198,47416,0.030492,1,1.0
4,2018-09-20,198,5944,0.053373,1,1.0



Train aggregated interactions
----------------------------------------------------------------------------------------------------
Shape: (19106319, 4)
Memory usage: 291.54 MB


,customer_idx,article_idx,purchase_count,implicit_weight
0,0,99,1.0,1.693147
1,0,16003,2.0,2.098612
2,0,23996,1.0,1.693147
3,0,29516,1.0,1.693147
4,0,30327,1.0,1.693147



Validation interactions
----------------------------------------------------------------------------------------------------
Shape: (4760953, 6)
Memory usage: 113.51 MB


,t_dat,customer_idx,article_idx,price,sales_channel_id,weight
0,2020-02-11,45,83608,0.040661,2,1.0
1,2020-02-11,45,88947,0.033898,2,1.0
2,2020-02-11,45,73556,0.013542,2,1.0
3,2020-02-11,309,44193,0.013542,2,1.0
4,2020-02-11,309,87849,0.010153,2,1.0



Test interactions
----------------------------------------------------------------------------------------------------
Shape: (4792094, 6)
Memory usage: 114.25 MB


,t_dat,customer_idx,article_idx,price,sales_channel_id,weight
0,2020-06-09,38,95122,0.033881,2,1.0
1,2020-06-09,38,95122,0.033881,2,1.0
2,2020-06-09,38,87885,0.025407,1,1.0
3,2020-06-09,38,94629,0.005068,1,1.0
4,2020-06-09,38,92358,0.042356,2,1.0


In [11]:
assert train_interactions["t_dat"].max() < val_interactions["t_dat"].min(), "Train/validation split overlap detected."
assert val_interactions["t_dat"].max() < test_interactions["t_dat"].min(), "Validation/test split overlap detected."

n_users = int(customer_id_map["customer_idx"].max()) + 1
n_items = int(article_id_map["article_idx"].max()) + 1

hybrid_dataset_audit = {
    "n_users": n_users,
    "n_items": n_items,
    "train_events": int(len(train_interactions)),
    "train_aggregated_pairs": int(len(train_interactions_aggregated)),
    "validation_events": int(len(val_interactions)),
    "test_events": int(len(test_interactions)),
    "train_date_min": str(train_interactions["t_dat"].min()),
    "train_date_max": str(train_interactions["t_dat"].max()),
    "validation_date_min": str(val_interactions["t_dat"].min()),
    "validation_date_max": str(val_interactions["t_dat"].max()),
    "test_date_min": str(test_interactions["t_dat"].min()),
    "test_date_max": str(test_interactions["t_dat"].max()),
}

print(json.dumps(hybrid_dataset_audit, indent=2))
save_json(hybrid_dataset_audit, HYBRID_ARTIFACTS_DIR / "hybrid_dataset_audit.json")

{
  "n_users": 1371980,
  "n_items": 105542,
  "train_events": 22235277,
  "train_aggregated_pairs": 19106319,
  "validation_events": 4760953,
  "test_events": 4792094,
  "train_date_min": "2018-09-20 00:00:00",
  "train_date_max": "2020-02-10 00:00:00",
  "validation_date_min": "2020-02-11 00:00:00",
  "validation_date_max": "2020-06-08 00:00:00",
  "test_date_min": "2020-06-09 00:00:00",
  "test_date_max": "2020-09-22 00:00:00"
}
Saved: /kaggle/working/hm-recommender/artifacts/hybrid_als_popularity/hybrid_dataset_audit.json


## 6. Load popularity signals

In [12]:
global_popularity_scores = pd.read_parquet(BASELINE_ARTIFACTS_DIR / "global_popularity_scores.parquet")
recency_weighted_popularity_scores = pd.read_parquet(BASELINE_ARTIFACTS_DIR / "recency_weighted_popularity_scores.parquet")
baseline_metrics = pd.read_csv(BASELINE_ARTIFACTS_DIR / "popularity_baseline_metrics.csv")

recent90_path = BASELINE_ARTIFACTS_DIR / "recent_90d_popularity_scores.parquet"
recent30_path = BASELINE_ARTIFACTS_DIR / "recent_30d_popularity_scores.parquet"

if recent90_path.exists():
    recent_popularity_scores = pd.read_parquet(recent90_path)
    recent_popularity_name = "recent_90d_popularity"
elif recent30_path.exists():
    recent_popularity_scores = pd.read_parquet(recent30_path)
    recent_popularity_name = "recent_30d_popularity"
else:
    print("No recent popularity score file found. Falling back to recency-weighted scores for the recent signal.")
    recent_popularity_scores = recency_weighted_popularity_scores.copy()
    recent_popularity_name = "recency_weighted_popularity_as_recent"

print_df_info(global_popularity_scores, "Global popularity scores")
print_df_info(recent_popularity_scores, f"{recent_popularity_name} scores")
print_df_info(recency_weighted_popularity_scores, "Recency-weighted popularity scores")

global_ranked_items = global_popularity_scores["article_idx"].to_numpy(dtype=np.int32)
recent_ranked_items = recent_popularity_scores["article_idx"].to_numpy(dtype=np.int32)
recency_ranked_items = recency_weighted_popularity_scores["article_idx"].to_numpy(dtype=np.int32)

best_popularity_row = (
    baseline_metrics[
        (baseline_metrics["eval_split"] == "validation")
        & (baseline_metrics["warm_start_only"] == True)
        & (baseline_metrics["k"] == PRIMARY_K)
    ]
    .sort_values(["recall_at_k", "ndcg_at_k", "mrr_at_k", "map_at_k"], ascending=False)
    .head(1)
)

print("Best popularity baseline for comparison:")
display(best_popularity_row)


Global popularity scores
----------------------------------------------------------------------------------------------------
Shape: (83275, 8)
Memory usage: 7.23 MB


,article_idx,popularity_score,purchase_count,unique_customers,last_train_purchase_date,avg_train_price,rank,article_id
0,53892,35373.0,35373,23701,2020-02-10,0.032287,1,0706016001
1,53893,26288.0,26288,19317,2020-02-10,0.032531,2,0706016002
2,1713,21688.0,21688,17997,2020-02-10,0.012916,3,0372860001
3,2236,19882.0,19882,14449,2020-02-10,0.030726,4,0399223001
4,14240,19642.0,19642,14369,2020-02-10,0.026013,5,0562245001



recent_90d_popularity scores
----------------------------------------------------------------------------------------------------
Shape: (83275, 9)
Memory usage: 8.82 MB


,article_idx,popularity_score,purchase_count,unique_customers,last_train_purchase_date,avg_train_price,rank,window_days,article_id
0,53892,10015.0,10015,7755,2020-02-10,0.031087,1,90,0706016001
1,58491,5411.0,5411,4768,2020-02-10,0.032285,2,90,0720125001
2,53893,5209.0,5209,4307,2020-02-10,0.031360,3,90,0706016002
3,73,4863.0,4863,2929,2020-02-10,0.011736,4,90,0158340001
4,14253,4719.0,4719,3933,2020-02-10,0.029468,5,90,0562245046



Recency-weighted popularity scores
----------------------------------------------------------------------------------------------------
Shape: (83275, 9)
Memory usage: 8.50 MB


,article_idx,popularity_score,purchase_count,unique_customers,last_train_purchase_date,avg_train_price,rank,half_life_days,article_id
0,53892,4019.874756,35373,23701,2020-02-10,0.032287,1,30.0,0706016001
1,58491,2614.516602,11755,9948,2020-02-10,0.032430,2,30.0,0720125001
2,53893,2354.024658,26288,19317,2020-02-10,0.032531,3,30.0,0706016002
3,9916,2091.709961,10352,8796,2020-02-10,0.015917,4,30.0,0537116001
4,3711,1998.038330,17655,13520,2020-02-10,0.016168,5,30.0,0464297007


Best popularity baseline for comparison:


,model_name,eval_split,warm_start_only,k,n_eval_users,precision_at_k,recall_at_k,map_at_k,ndcg_at_k,mrr_at_k,hitrate_at_k,unique_recommended_items,elapsed_seconds,catalog_coverage
3,global_popularity,validation,True,20,450725,0.004488,0.014219,0.003822,0.01007,0.020863,0.080044,32,14.21,0.000384


In [13]:
def make_normalized_score_array(scores_df: pd.DataFrame, n_items: int, score_col: str = "popularity_score") -> np.ndarray:
    """Create item-indexed normalized score array in [0, 1] using log1p popularity."""
    arr = np.zeros(n_items, dtype=np.float32)

    if score_col not in scores_df.columns:
        raise ValueError(f"{score_col} not found in scores dataframe columns: {scores_df.columns.tolist()}")

    item_idx = scores_df["article_idx"].to_numpy(dtype=np.int32)
    raw_scores = scores_df[score_col].fillna(0).to_numpy(dtype=np.float32)
    log_scores = np.log1p(np.maximum(raw_scores, 0))

    max_score = float(log_scores.max()) if len(log_scores) else 0.0
    if max_score > 0:
        norm_scores = log_scores / max_score
    else:
        norm_scores = log_scores

    arr[item_idx] = norm_scores.astype(np.float32)
    return arr


global_score_arr = make_normalized_score_array(global_popularity_scores, n_items)
recent90_score_arr = make_normalized_score_array(recent_popularity_scores, n_items)
recency_score_arr = make_normalized_score_array(recency_weighted_popularity_scores, n_items)

popularity_signal_audit = {
    "global_score_max": float(global_score_arr.max()),
    "recent_signal_name": recent_popularity_name,
    "recent_score_max": float(recent90_score_arr.max()),
    "recency_weighted_score_max": float(recency_score_arr.max()),
    "global_ranked_items": int(len(global_ranked_items)),
    "recent_ranked_items": int(len(recent_ranked_items)),
    "recency_ranked_items": int(len(recency_ranked_items)),
}

print(json.dumps(popularity_signal_audit, indent=2))
save_json(popularity_signal_audit, HYBRID_ARTIFACTS_DIR / "popularity_signal_audit.json")

{
  "global_score_max": 1.0,
  "recent_signal_name": "recent_90d_popularity",
  "recent_score_max": 1.0,
  "recency_weighted_score_max": 1.0,
  "global_ranked_items": 83275,
  "recent_ranked_items": 83275,
  "recency_ranked_items": 83275
}
Saved: /kaggle/working/hm-recommender/artifacts/hybrid_als_popularity/popularity_signal_audit.json


## 7. Load best tuned ALS config and build matrix

In [14]:
if BEST_TUNING_CONFIG_PATH is not None and BEST_TUNING_CONFIG_PATH.exists():
    with open(BEST_TUNING_CONFIG_PATH) as f:
        best_als_config = json.load(f)
    print("Loaded best tuned ALS config from:", BEST_TUNING_CONFIG_PATH)
else:
    best_als_config = FALLBACK_BEST_ALS_CONFIG.copy()
    print("Using fallback best tuned ALS config from previous review.")

# Normalize key names/types.
best_als_config["factors"] = int(best_als_config["factors"])
best_als_config["alpha"] = float(best_als_config["alpha"])
best_als_config["regularization"] = float(best_als_config["regularization"])
best_als_config["iterations"] = int(best_als_config["iterations"])
best_als_config.setdefault(
    "model_name",
    f"als_f{best_als_config['factors']}_reg{best_als_config['regularization']}_it{best_als_config['iterations']}_alpha{best_als_config['alpha']}"
)

print("Best ALS config for hybrid:")
print(json.dumps(best_als_config, indent=2))
save_json(best_als_config, HYBRID_ARTIFACTS_DIR / "hybrid_base_als_config.json")

Using fallback best tuned ALS config from previous review.
Best ALS config for hybrid:
{
  "model_name": "als_f64_reg0p1_it15_alpha20p0",
  "stage": "main_grid",
  "factors": 64,
  "alpha": 20.0,
  "regularization": 0.1,
  "iterations": 15
}
Saved: /kaggle/working/hm-recommender/artifacts/hybrid_als_popularity/hybrid_base_als_config.json


In [15]:
row_indices = train_interactions_aggregated["customer_idx"].to_numpy(dtype=np.int32)
col_indices = train_interactions_aggregated["article_idx"].to_numpy(dtype=np.int32)
values = train_interactions_aggregated["implicit_weight"].to_numpy(dtype=np.float32)

user_items = csr_matrix(
    (values, (row_indices, col_indices)),
    shape=(n_users, n_items),
    dtype=np.float32,
)
user_items.eliminate_zeros()

print("user_items shape:", user_items.shape)
print("user_items non-zero entries:", user_items.nnz)
print("density percent:", user_items.nnz / (user_items.shape[0] * user_items.shape[1]) * 100)

save_npz(HYBRID_ARTIFACTS_DIR / "hybrid_train_user_items_matrix.npz", user_items)
print("Saved sparse matrix:", HYBRID_ARTIFACTS_DIR / "hybrid_train_user_items_matrix.npz")

user_items shape: (1371980, 105542)
user_items non-zero entries: 19106319
density percent: 0.013194833799069671
Saved sparse matrix: /kaggle/working/hm-recommender/artifacts/hybrid_als_popularity/hybrid_train_user_items_matrix.npz


## 8. Train best tuned ALS model

In [16]:
def train_als_model(config: dict, user_items_matrix: csr_matrix) -> AlternatingLeastSquares:
    print("=" * 120)
    print("Training base ALS model for hybrid:", config["model_name"])
    print(config)

    model = AlternatingLeastSquares(
        factors=int(config["factors"]),
        regularization=float(config["regularization"]),
        iterations=int(config["iterations"]),
        alpha=float(config["alpha"]),
        random_state=RANDOM_SEED,
        use_gpu=False,
        calculate_training_loss=False,
    )

    started = time.time()
    model.fit(user_items_matrix, show_progress=True)
    elapsed = time.time() - started
    print(f"Training completed in {elapsed:.2f} seconds.")
    return model, elapsed


base_als_model, base_als_train_seconds = train_als_model(best_als_config, user_items)

model_dir = HYBRID_ARTIFACTS_DIR / "base_als_model"
model_dir.mkdir(parents=True, exist_ok=True)
np.save(model_dir / "user_factors.npy", base_als_model.user_factors)
np.save(model_dir / "item_factors.npy", base_als_model.item_factors)
save_json(best_als_config, model_dir / "base_als_config.json")

print("Saved base ALS factor arrays to:", model_dir)

Training base ALS model for hybrid: als_f64_reg0p1_it15_alpha20p0
{'model_name': 'als_f64_reg0p1_it15_alpha20p0', 'stage': 'main_grid', 'factors': 64, 'alpha': 20.0, 'regularization': 0.1, 'iterations': 15}


/usr/local/lib/python3.12/dist-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 4 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/15 [00:00<?, ?it/s]

Training completed in 131.30 seconds.
Saved: /kaggle/working/hm-recommender/artifacts/hybrid_als_popularity/base_als_model/base_als_config.json
Saved base ALS factor arrays to: /kaggle/working/hm-recommender/artifacts/hybrid_als_popularity/base_als_model


## 9. Build evaluation dictionaries

In [17]:
print("Building train seen and validation/test target dictionaries...")
started = time.time()
train_seen_by_user = build_user_item_sets(train_interactions)
val_actual_by_user = build_user_item_sets(val_interactions)
test_actual_by_user = build_user_item_sets(test_interactions)
print(f"Built dictionaries in {time.time() - started:.2f} seconds.")

val_warm_users = [int(u) for u in val_actual_by_user.keys() if u in train_seen_by_user]
test_warm_users = [int(u) for u in test_actual_by_user.keys() if u in train_seen_by_user]

hybrid_split_audit = {
    "train_seen_users": len(train_seen_by_user),
    "validation_target_users": len(val_actual_by_user),
    "validation_warm_users": len(val_warm_users),
    "validation_cold_users": len(val_actual_by_user) - len(val_warm_users),
    "test_target_users": len(test_actual_by_user),
    "test_warm_users": len(test_warm_users),
    "test_cold_users": len(test_actual_by_user) - len(test_warm_users),
    "hybrid_tuning_max_eval_users": HYBRID_TUNING_MAX_EVAL_USERS,
}

print(json.dumps(hybrid_split_audit, indent=2))
save_json(hybrid_split_audit, HYBRID_ARTIFACTS_DIR / "hybrid_split_audit.json")

Building train seen and validation/test target dictionaries...
Built dictionaries in 63.31 seconds.
{
  "train_seen_users": 1150557,
  "validation_target_users": 574129,
  "validation_warm_users": 450725,
  "validation_cold_users": 123404,
  "test_target_users": 578785,
  "test_warm_users": 456034,
  "test_cold_users": 122751,
  "hybrid_tuning_max_eval_users": 50000
}
Saved: /kaggle/working/hm-recommender/artifacts/hybrid_als_popularity/hybrid_split_audit.json


## 10. Hybrid recommendation helpers

In [18]:
def batch_recommend_als_with_scores(
    model: AlternatingLeastSquares,
    user_items_matrix: csr_matrix,
    user_ids: List[int],
    n_candidates: int,
    batch_size: int = 4096,
) -> Iterable[Tuple[int, np.ndarray, np.ndarray]]:
    """Yield (user_id, item_ids, scores) using implicit batch recommend with fallback."""
    for start in range(0, len(user_ids), batch_size):
        batch_users = np.array(user_ids[start:start + batch_size], dtype=np.int32)
        if len(batch_users) == 0:
            continue

        try:
            ids, scores = model.recommend(
                batch_users,
                user_items_matrix[batch_users],
                N=n_candidates,
                filter_already_liked_items=True,
                recalculate_user=False,
            )
            for user_id, rec_items, rec_scores in zip(batch_users, ids, scores):
                valid_mask = rec_items >= 0
                yield int(user_id), rec_items[valid_mask].astype(np.int32), rec_scores[valid_mask].astype(np.float32)
        except Exception as batch_error:
            print("Batch recommend failed once; falling back to per-user recommendation.")
            print("Batch error:", repr(batch_error))
            for user_id in batch_users:
                rec_items, rec_scores = model.recommend(
                    int(user_id),
                    user_items_matrix[int(user_id)],
                    N=n_candidates,
                    filter_already_liked_items=True,
                    recalculate_user=False,
                )
                valid_mask = rec_items >= 0
                yield int(user_id), rec_items[valid_mask].astype(np.int32), rec_scores[valid_mask].astype(np.float32)


def normalize_als_scores(scores: np.ndarray) -> np.ndarray:
    if len(scores) == 0:
        return scores.astype(np.float32)
    scores = scores.astype(np.float32)
    min_score = float(scores.min())
    max_score = float(scores.max())
    if max_score - min_score < 1e-8:
        return np.ones_like(scores, dtype=np.float32)
    return ((scores - min_score) / (max_score - min_score)).astype(np.float32)


def dedupe_preserve_order(items: Iterable[int]) -> List[int]:
    seen = set()
    output = []
    for item in items:
        item_int = int(item)
        if item_int not in seen:
            seen.add(item_int)
            output.append(item_int)
    return output


def build_candidate_items(
    als_items: np.ndarray,
    seen_items: Optional[set],
    global_ranked_items: np.ndarray,
    recent_ranked_items: np.ndarray,
    recency_ranked_items: np.ndarray,
    popularity_candidates: int,
) -> List[int]:
    candidate_items = []
    candidate_items.extend([int(x) for x in als_items])
    candidate_items.extend([int(x) for x in global_ranked_items[:popularity_candidates]])
    candidate_items.extend([int(x) for x in recent_ranked_items[:popularity_candidates]])
    candidate_items.extend([int(x) for x in recency_ranked_items[:popularity_candidates]])

    deduped = dedupe_preserve_order(candidate_items)

    if seen_items:
        deduped = [item for item in deduped if item not in seen_items]

    return deduped


def score_hybrid_candidates(
    candidate_items: List[int],
    als_items: np.ndarray,
    als_scores: np.ndarray,
    config: dict,
    global_score_arr: np.ndarray,
    recent90_score_arr: np.ndarray,
    recency_score_arr: np.ndarray,
    k: int,
) -> List[int]:
    if not candidate_items:
        return []

    als_norm_scores = normalize_als_scores(als_scores)
    als_score_map = {int(item): float(score) for item, score in zip(als_items, als_norm_scores)}

    scored = []
    for order, item in enumerate(candidate_items):
        if item < 0 or item >= len(global_score_arr):
            continue

        final_score = (
            config["als"] * als_score_map.get(item, 0.0)
            + config["global"] * float(global_score_arr[item])
            + config["recent90"] * float(recent90_score_arr[item])
            + config["recency"] * float(recency_score_arr[item])
        )
        scored.append((final_score, -order, item))

    scored.sort(reverse=True)
    return [int(item) for _, _, item in scored[:k]]

In [19]:
def evaluate_hybrid_configs(
    model: AlternatingLeastSquares,
    model_name: str,
    user_items_matrix: csr_matrix,
    eval_actual_by_user: Dict[int, set],
    eval_user_ids: List[int],
    hybrid_configs: List[dict],
    eval_name: str,
    k_values: List[int],
    max_eval_users: Optional[int],
    total_catalog_items: int,
) -> Tuple[pd.DataFrame, Dict[str, Dict[int, List[int]]]]:
    started = time.time()
    max_k = max(k_values)
    eval_user_ids = sample_eval_users(eval_user_ids, max_eval_users)

    if not eval_user_ids:
        raise ValueError(f"No evaluation users found for {eval_name}.")

    sums_by_config = {
        cfg["hybrid_name"]: {
            k: {"precision": 0.0, "recall": 0.0, "map": 0.0, "ndcg": 0.0, "mrr": 0.0, "hitrate": 0.0}
            for k in k_values
        }
        for cfg in hybrid_configs
    }
    unique_by_config = {cfg["hybrid_name"]: {k: set() for k in k_values} for cfg in hybrid_configs}
    sample_recs_by_config = {cfg["hybrid_name"]: {} for cfg in hybrid_configs}

    scored_users = 0
    for user_id, als_items, als_scores in batch_recommend_als_with_scores(
        model=model,
        user_items_matrix=user_items_matrix,
        user_ids=eval_user_ids,
        n_candidates=ALS_CANDIDATES,
        batch_size=RECOMMEND_BATCH_SIZE,
    ):
        actual = eval_actual_by_user[user_id]
        seen_items = train_seen_by_user.get(user_id)

        candidate_items = build_candidate_items(
            als_items=als_items,
            seen_items=seen_items,
            global_ranked_items=global_ranked_items,
            recent_ranked_items=recent_ranked_items,
            recency_ranked_items=recency_ranked_items,
            popularity_candidates=POPULARITY_CANDIDATES,
        )

        for cfg in hybrid_configs:
            hybrid_name = cfg["hybrid_name"]
            recommended = score_hybrid_candidates(
                candidate_items=candidate_items,
                als_items=als_items,
                als_scores=als_scores,
                config=cfg,
                global_score_arr=global_score_arr,
                recent90_score_arr=recent90_score_arr,
                recency_score_arr=recency_score_arr,
                k=max_k,
            )

            # Safety fallback. This should rarely be needed.
            if len(recommended) < max_k:
                fallback_extra = recommend_from_ranked_items(global_ranked_items, seen_items, max_k, n_candidates=5000)
                recommended = dedupe_preserve_order(recommended + fallback_extra)[:max_k]

            if len(sample_recs_by_config[hybrid_name]) < 10:
                sample_recs_by_config[hybrid_name][user_id] = recommended

            update_metric_sums(
                recommended=recommended,
                actual=actual,
                k_values=k_values,
                sums=sums_by_config[hybrid_name],
                unique_recommended_items_by_k=unique_by_config[hybrid_name],
            )

        scored_users += 1

    elapsed_seconds = time.time() - started
    rows = []

    for cfg in hybrid_configs:
        hybrid_name = cfg["hybrid_name"]
        for k in k_values:
            rows.append(
                {
                    "model_name": hybrid_name,
                    "model_family": "hybrid_als_popularity",
                    "base_als_model_name": model_name,
                    "eval_split": eval_name,
                    "warm_start_only": True,
                    "k": k,
                    "n_eval_users": scored_users,
                    "als_weight": cfg["als"],
                    "global_weight": cfg["global"],
                    "recent90_weight": cfg["recent90"],
                    "recency_weight": cfg["recency"],
                    "als_candidates": ALS_CANDIDATES,
                    "popularity_candidates": POPULARITY_CANDIDATES,
                    "precision_at_k": sums_by_config[hybrid_name][k]["precision"] / scored_users,
                    "recall_at_k": sums_by_config[hybrid_name][k]["recall"] / scored_users,
                    "map_at_k": sums_by_config[hybrid_name][k]["map"] / scored_users,
                    "ndcg_at_k": sums_by_config[hybrid_name][k]["ndcg"] / scored_users,
                    "mrr_at_k": sums_by_config[hybrid_name][k]["mrr"] / scored_users,
                    "hitrate_at_k": sums_by_config[hybrid_name][k]["hitrate"] / scored_users,
                    "unique_recommended_items": len(unique_by_config[hybrid_name][k]),
                    "catalog_coverage": len(unique_by_config[hybrid_name][k]) / total_catalog_items if total_catalog_items > 0 else np.nan,
                    "elapsed_seconds": round(elapsed_seconds, 2),
                    "max_eval_users": max_eval_users,
                }
            )

    metrics_df = pd.DataFrame(rows)
    metric_cols = ["precision_at_k", "recall_at_k", "map_at_k", "ndcg_at_k", "mrr_at_k", "hitrate_at_k", "catalog_coverage"]
    metrics_df[metric_cols] = metrics_df[metric_cols].round(6)
    return metrics_df, sample_recs_by_config

## 11. Tune hybrid weights on validation warm-start users

In [20]:
total_catalog_items = int(train_interactions["article_idx"].nunique())

hybrid_tuning_metrics, hybrid_tuning_samples = evaluate_hybrid_configs(
    model=base_als_model,
    model_name=best_als_config["model_name"],
    user_items_matrix=user_items,
    eval_actual_by_user=val_actual_by_user,
    eval_user_ids=val_warm_users,
    hybrid_configs=HYBRID_WEIGHT_CONFIGS,
    eval_name="validation_tuning_sample",
    k_values=TOP_K_VALUES,
    max_eval_users=HYBRID_TUNING_MAX_EVAL_USERS,
    total_catalog_items=total_catalog_items,
)

hybrid_tuning_leaderboard = (
    hybrid_tuning_metrics[hybrid_tuning_metrics["k"] == PRIMARY_K]
    .sort_values(["recall_at_k", "ndcg_at_k", "mrr_at_k", "map_at_k", "catalog_coverage"], ascending=False)
    .reset_index(drop=True)
)

print("Hybrid tuning leaderboard at K=20:")
display(hybrid_tuning_leaderboard)

save_csv(hybrid_tuning_metrics, HYBRID_ARTIFACTS_DIR / "hybrid_tuning_metrics.csv")
save_csv(hybrid_tuning_leaderboard, HYBRID_ARTIFACTS_DIR / "hybrid_tuning_leaderboard_k20.csv")

best_hybrid_names = hybrid_tuning_leaderboard.head(FINAL_TOP_N_HYBRIDS)["model_name"].tolist()
best_hybrid_configs = [cfg for cfg in HYBRID_WEIGHT_CONFIGS if cfg["hybrid_name"] in set(best_hybrid_names)]

best_hybrid_config = best_hybrid_configs[0]
print("Best hybrid config from tuning:")
print(json.dumps(best_hybrid_config, indent=2))
save_json(best_hybrid_config, HYBRID_ARTIFACTS_DIR / "best_hybrid_config.json")

Hybrid tuning leaderboard at K=20:


,model_name,model_family,base_als_model_name,eval_split,warm_start_only,k,n_eval_users,als_weight,global_weight,recent90_weight,recency_weight,als_candidates,popularity_candidates,precision_at_k,recall_at_k,map_at_k,ndcg_at_k,mrr_at_k,hitrate_at_k,unique_recommended_items,catalog_coverage,elapsed_seconds,max_eval_users
0,hybrid_als40_global40_recent20,hybrid_als_popularity,als_f64_reg0p1_it15_alpha20p0,validation_tuning_sample,True,20,50000,0.4,0.40,0.20,0.0,100,200,0.005161,0.016748,0.004987,0.012265,0.024854,0.08762,3685,0.044251,301.38,50000
1,hybrid_als50_global25_recent15_recency10,hybrid_als_popularity,als_f64_reg0p1_it15_alpha20p0,validation_tuning_sample,True,20,50000,0.5,0.25,0.15,0.1,100,200,0.005046,0.016421,0.004900,0.012022,0.024350,0.08546,4167,0.050039,301.38,50000
2,hybrid_als50_global30_recent20,hybrid_als_popularity,als_f64_reg0p1_it15_alpha20p0,validation_tuning_sample,True,20,50000,0.5,0.30,0.20,0.0,100,200,0.004925,0.016087,0.004825,0.011793,0.023914,0.08336,4246,0.050988,301.38,50000
3,hybrid_als60_global20_recency20,hybrid_als_popularity,als_f64_reg0p1_it15_alpha20p0,validation_tuning_sample,True,20,50000,0.6,0.20,0.00,0.2,100,200,0.004808,0.015762,0.004725,0.011538,0.023399,0.08154,4577,0.054962,301.38,50000
4,hybrid_als60_global25_recent15,hybrid_als_popularity,als_f64_reg0p1_it15_alpha20p0,validation_tuning_sample,True,20,50000,0.6,0.25,0.15,0.0,100,200,0.004712,0.015417,0.004647,0.011326,0.023041,0.08002,4625,0.055539,301.38,50000
5,hybrid_als70_global20_recent10,hybrid_als_popularity,als_f64_reg0p1_it15_alpha20p0,validation_tuning_sample,True,20,50000,0.7,0.20,0.10,0.0,100,200,0.004519,0.014825,0.004523,0.010949,0.022404,0.07700,4857,0.058325,301.38,50000
6,hybrid_als50_global50,hybrid_als_popularity,als_f64_reg0p1_it15_alpha20p0,validation_tuning_sample,True,20,50000,0.5,0.50,0.00,0.0,100,200,0.004483,0.014545,0.004494,0.010867,0.022460,0.07658,4539,0.054506,301.38,50000
7,hybrid_als60_global40,hybrid_als_popularity,als_f64_reg0p1_it15_alpha20p0,validation_tuning_sample,True,20,50000,0.6,0.40,0.00,0.0,100,200,0.004388,0.014226,0.004430,0.010667,0.022084,0.07486,4764,0.057208,301.38,50000
8,hybrid_als70_global30,hybrid_als_popularity,als_f64_reg0p1_it15_alpha20p0,validation_tuning_sample,True,20,50000,0.7,0.30,0.00,0.0,100,200,0.004314,0.014059,0.004368,0.010517,0.021780,0.07384,4929,0.059189,301.38,50000
9,hybrid_als80_global20,hybrid_als_popularity,als_f64_reg0p1_it15_alpha20p0,validation_tuning_sample,True,20,50000,0.8,0.20,0.00,0.0,100,200,0.004268,0.014013,0.004313,0.010418,0.021540,0.07326,5057,0.060727,301.38,50000


Saved: /kaggle/working/hm-recommender/artifacts/hybrid_als_popularity/hybrid_tuning_metrics.csv | 0.00 MB
Saved: /kaggle/working/hm-recommender/artifacts/hybrid_als_popularity/hybrid_tuning_leaderboard_k20.csv | 0.00 MB
Best hybrid config from tuning:
{
  "hybrid_name": "hybrid_als40_global40_recent20",
  "als": 0.4,
  "global": 0.4,
  "recent90": 0.2,
  "recency": 0.0
}
Saved: /kaggle/working/hm-recommender/artifacts/hybrid_als_popularity/best_hybrid_config.json


## 12. Final full evaluation of best hybrid

In [21]:
final_metric_frames = []
final_sample_recommendations = {}

if RUN_FULL_BEST_HYBRID_EVALUATION:
    for eval_name, actual_by_user, warm_users in [
        ("validation", val_actual_by_user, val_warm_users),
        ("test", test_actual_by_user, test_warm_users),
    ]:
        print()
        print(f"Final evaluation for {eval_name} using top {len(best_hybrid_configs)} hybrid config(s).")
        metrics_df, sample_recs = evaluate_hybrid_configs(
            model=base_als_model,
            model_name=best_als_config["model_name"],
            user_items_matrix=user_items,
            eval_actual_by_user=actual_by_user,
            eval_user_ids=warm_users,
            hybrid_configs=best_hybrid_configs,
            eval_name=eval_name,
            k_values=TOP_K_VALUES,
            max_eval_users=FINAL_MAX_EVAL_USERS,
            total_catalog_items=total_catalog_items,
        )
        display(metrics_df)
        final_metric_frames.append(metrics_df)
        final_sample_recommendations[eval_name] = sample_recs

    final_hybrid_metrics = pd.concat(final_metric_frames, ignore_index=True)
else:
    print("Skipping full final evaluation. Using tuning-sample metrics as final_hybrid_metrics.")
    final_hybrid_metrics = hybrid_tuning_metrics.copy()

final_hybrid_leaderboard = (
    final_hybrid_metrics[final_hybrid_metrics["k"] == PRIMARY_K]
    .sort_values(["eval_split", "recall_at_k", "ndcg_at_k", "mrr_at_k", "map_at_k"], ascending=[True, False, False, False, False])
    .reset_index(drop=True)
)

print("Final hybrid leaderboard at K=20:")
display(final_hybrid_leaderboard)

save_csv(final_hybrid_metrics, HYBRID_ARTIFACTS_DIR / "final_hybrid_metrics.csv")
save_csv(final_hybrid_leaderboard, HYBRID_ARTIFACTS_DIR / "final_hybrid_leaderboard_k20.csv")


Final evaluation for validation using top 1 hybrid config(s).


,model_name,model_family,base_als_model_name,eval_split,warm_start_only,k,n_eval_users,als_weight,global_weight,recent90_weight,recency_weight,als_candidates,popularity_candidates,precision_at_k,recall_at_k,map_at_k,ndcg_at_k,mrr_at_k,hitrate_at_k,unique_recommended_items,catalog_coverage,elapsed_seconds,max_eval_users
0,hybrid_als40_global40_recent20,hybrid_als_popularity,als_f64_reg0p1_it15_alpha20p0,validation,True,12,450725,0.4,0.4,0.2,0.0,100,200,0.006115,0.011990,0.004896,0.010855,0.023580,0.064654,3897,0.046797,597.01,None
1,hybrid_als40_global40_recent20,hybrid_als_popularity,als_f64_reg0p1_it15_alpha20p0,validation,True,20,450725,0.4,0.4,0.2,0.0,100,200,0.005247,0.016886,0.004994,0.012370,0.025138,0.089602,4464,0.053606,597.01,None



Final evaluation for test using top 1 hybrid config(s).


,model_name,model_family,base_als_model_name,eval_split,warm_start_only,k,n_eval_users,als_weight,global_weight,recent90_weight,recency_weight,als_candidates,popularity_candidates,precision_at_k,recall_at_k,map_at_k,ndcg_at_k,mrr_at_k,hitrate_at_k,unique_recommended_items,catalog_coverage,elapsed_seconds,max_eval_users
0,hybrid_als40_global40_recent20,hybrid_als_popularity,als_f64_reg0p1_it15_alpha20p0,test,True,12,456034,0.4,0.4,0.2,0.0,100,200,0.004398,0.008630,0.003437,0.007785,0.017238,0.047720,3892,0.046737,608.92,None
1,hybrid_als40_global40_recent20,hybrid_als_popularity,als_f64_reg0p1_it15_alpha20p0,test,True,20,456034,0.4,0.4,0.2,0.0,100,200,0.003754,0.012055,0.003499,0.008852,0.018396,0.066291,4471,0.053690,608.92,None


Final hybrid leaderboard at K=20:


,model_name,model_family,base_als_model_name,eval_split,warm_start_only,k,n_eval_users,als_weight,global_weight,recent90_weight,recency_weight,als_candidates,popularity_candidates,precision_at_k,recall_at_k,map_at_k,ndcg_at_k,mrr_at_k,hitrate_at_k,unique_recommended_items,catalog_coverage,elapsed_seconds,max_eval_users
0,hybrid_als40_global40_recent20,hybrid_als_popularity,als_f64_reg0p1_it15_alpha20p0,test,True,20,456034,0.4,0.4,0.2,0.0,100,200,0.003754,0.012055,0.003499,0.008852,0.018396,0.066291,4471,0.053690,608.92,None
1,hybrid_als40_global40_recent20,hybrid_als_popularity,als_f64_reg0p1_it15_alpha20p0,validation,True,20,450725,0.4,0.4,0.2,0.0,100,200,0.005247,0.016886,0.004994,0.012370,0.025138,0.089602,4464,0.053606,597.01,None


Saved: /kaggle/working/hm-recommender/artifacts/hybrid_als_popularity/final_hybrid_metrics.csv | 0.00 MB
Saved: /kaggle/working/hm-recommender/artifacts/hybrid_als_popularity/final_hybrid_leaderboard_k20.csv | 0.00 MB


## 13. Compare popularity, tuned ALS, and hybrid

In [22]:
comparison_rows = []

# Best popularity baseline row.
if len(best_popularity_row) > 0:
    row = best_popularity_row.iloc[0].to_dict()
    comparison_rows.append(
        {
            "model_family": "best_popularity_baseline",
            "model_name": row.get("model_name", "unknown"),
            "eval_split": row.get("eval_split", "validation"),
            "warm_start_only": row.get("warm_start_only", True),
            "k": row.get("k", PRIMARY_K),
            "precision_at_k": row.get("precision_at_k", np.nan),
            "recall_at_k": row.get("recall_at_k", np.nan),
            "map_at_k": row.get("map_at_k", np.nan),
            "ndcg_at_k": row.get("ndcg_at_k", np.nan),
            "mrr_at_k": row.get("mrr_at_k", np.nan),
            "hitrate_at_k": row.get("hitrate_at_k", np.nan),
            "catalog_coverage": row.get("catalog_coverage", np.nan),
        }
    )

# Best tuned ALS row from notebook 05, if attached.
if TUNING_COMPARISON_PATH is not None and Path(TUNING_COMPARISON_PATH).exists():
    tuning_comparison = pd.read_csv(TUNING_COMPARISON_PATH)
    tuned_als_rows = tuning_comparison[tuning_comparison["model_family"] == "best_tuned_als"]
    if len(tuned_als_rows) > 0:
        row = tuned_als_rows.iloc[0].to_dict()
        comparison_rows.append(
            {
                "model_family": "best_tuned_als_from_05",
                "model_name": row.get("model_name", best_als_config["model_name"]),
                "eval_split": "validation_tuning_sample",
                "warm_start_only": True,
                "k": row.get("k", PRIMARY_K),
                "precision_at_k": row.get("precision_at_k", np.nan),
                "recall_at_k": row.get("recall_at_k", np.nan),
                "map_at_k": row.get("map_at_k", np.nan),
                "ndcg_at_k": row.get("ndcg_at_k", np.nan),
                "mrr_at_k": row.get("mrr_at_k", np.nan),
                "hitrate_at_k": row.get("hitrate_at_k", np.nan),
                "catalog_coverage": row.get("catalog_coverage", np.nan),
            }
        )

# Hybrid tuning winner.
hybrid_tuning_winner = hybrid_tuning_leaderboard.iloc[0].to_dict()
comparison_rows.append(
    {
        "model_family": "best_hybrid_tuning_sample",
        "model_name": hybrid_tuning_winner["model_name"],
        "eval_split": hybrid_tuning_winner["eval_split"],
        "warm_start_only": hybrid_tuning_winner["warm_start_only"],
        "k": hybrid_tuning_winner["k"],
        "precision_at_k": hybrid_tuning_winner["precision_at_k"],
        "recall_at_k": hybrid_tuning_winner["recall_at_k"],
        "map_at_k": hybrid_tuning_winner["map_at_k"],
        "ndcg_at_k": hybrid_tuning_winner["ndcg_at_k"],
        "mrr_at_k": hybrid_tuning_winner["mrr_at_k"],
        "hitrate_at_k": hybrid_tuning_winner["hitrate_at_k"],
        "catalog_coverage": hybrid_tuning_winner["catalog_coverage"],
    }
)

# Final full hybrid rows.
for row in final_hybrid_metrics[(final_hybrid_metrics["k"] == PRIMARY_K)].to_dict(orient="records"):
    comparison_rows.append(
        {
            "model_family": "best_hybrid_final",
            "model_name": row["model_name"],
            "eval_split": row["eval_split"],
            "warm_start_only": row["warm_start_only"],
            "k": row["k"],
            "precision_at_k": row["precision_at_k"],
            "recall_at_k": row["recall_at_k"],
            "map_at_k": row["map_at_k"],
            "ndcg_at_k": row["ndcg_at_k"],
            "mrr_at_k": row["mrr_at_k"],
            "hitrate_at_k": row["hitrate_at_k"],
            "catalog_coverage": row["catalog_coverage"],
        }
    )

model_comparison = pd.DataFrame(comparison_rows)
metric_cols = ["precision_at_k", "recall_at_k", "map_at_k", "ndcg_at_k", "mrr_at_k", "hitrate_at_k", "catalog_coverage"]
model_comparison[metric_cols] = model_comparison[metric_cols].apply(pd.to_numeric, errors="coerce").round(6)

display(model_comparison)
save_csv(model_comparison, HYBRID_ARTIFACTS_DIR / "hybrid_model_comparison.csv")

# Decision flags comparing validation final hybrid against global popularity.
validation_final_rows = model_comparison[
    (model_comparison["model_family"] == "best_hybrid_final")
    & (model_comparison["eval_split"] == "validation")
]

if len(best_popularity_row) > 0 and len(validation_final_rows) > 0:
    pop = best_popularity_row.iloc[0]
    hyb = validation_final_rows.sort_values(["recall_at_k", "ndcg_at_k", "mrr_at_k"], ascending=False).iloc[0]
    decision_summary = {
        "best_hybrid_model": hyb["model_name"],
        "best_popularity_model": pop["model_name"],
        "hybrid_beats_popularity_recall_at_20": bool(hyb["recall_at_k"] > pop["recall_at_k"]),
        "hybrid_beats_popularity_ndcg_at_20": bool(hyb["ndcg_at_k"] > pop["ndcg_at_k"]),
        "hybrid_beats_popularity_mrr_at_20": bool(hyb["mrr_at_k"] > pop["mrr_at_k"]),
        "hybrid_beats_popularity_map_at_20": bool(hyb["map_at_k"] > pop["map_at_k"]),
        "hybrid_beats_popularity_coverage_at_20": bool(hyb["catalog_coverage"] > pop["catalog_coverage"]),
        "best_hybrid_validation_row": hyb.to_dict(),
        "best_popularity_validation_row": pop.to_dict(),
    }
else:
    decision_summary = {"warning": "Could not compare final hybrid against popularity."}

print(json.dumps(decision_summary, indent=2, default=str))
save_json(decision_summary, HYBRID_ARTIFACTS_DIR / "hybrid_decision_summary.json")

,model_family,model_name,eval_split,warm_start_only,k,precision_at_k,recall_at_k,map_at_k,ndcg_at_k,mrr_at_k,hitrate_at_k,catalog_coverage
0,best_popularity_baseline,global_popularity,validation,True,20,0.004488,0.014219,0.003822,0.010070,0.020863,0.080044,0.000384
1,best_hybrid_tuning_sample,hybrid_als40_global40_recent20,validation_tuning_sample,True,20,0.005161,0.016748,0.004987,0.012265,0.024854,0.087620,0.044251
2,best_hybrid_final,hybrid_als40_global40_recent20,validation,True,20,0.005247,0.016886,0.004994,0.012370,0.025138,0.089602,0.053606
3,best_hybrid_final,hybrid_als40_global40_recent20,test,True,20,0.003754,0.012055,0.003499,0.008852,0.018396,0.066291,0.053690


Saved: /kaggle/working/hm-recommender/artifacts/hybrid_als_popularity/hybrid_model_comparison.csv | 0.00 MB
{
  "best_hybrid_model": "hybrid_als40_global40_recent20",
  "best_popularity_model": "global_popularity",
  "hybrid_beats_popularity_recall_at_20": true,
  "hybrid_beats_popularity_ndcg_at_20": true,
  "hybrid_beats_popularity_mrr_at_20": true,
  "hybrid_beats_popularity_map_at_20": true,
  "hybrid_beats_popularity_coverage_at_20": true,
  "best_hybrid_validation_row": {
    "model_family": "best_hybrid_final",
    "model_name": "hybrid_als40_global40_recent20",
    "eval_split": "validation",
    "warm_start_only": true,
    "k": 20,
    "precision_at_k": 0.005247,
    "recall_at_k": 0.016886,
    "map_at_k": 0.004994,
    "ndcg_at_k": 0.01237,
    "mrr_at_k": 0.025138,
    "hitrate_at_k": 0.089602,
    "catalog_coverage": 0.053606
  },
  "best_popularity_validation_row": {
    "model_name": "global_popularity",
    "eval_split": "validation",
    "warm_start_only": true,
    "

## 14. Demo hybrid recommendations

In [23]:
def recommend_hybrid_for_users(
    user_ids: Iterable[int],
    model: AlternatingLeastSquares,
    model_name: str,
    hybrid_config: dict,
    k: int,
) -> pd.DataFrame:
    user_ids = [int(u) for u in user_ids if int(u) in train_seen_by_user]
    rows = []

    for user_id, als_items, als_scores in batch_recommend_als_with_scores(
        model=model,
        user_items_matrix=user_items,
        user_ids=user_ids,
        n_candidates=ALS_CANDIDATES,
        batch_size=RECOMMEND_BATCH_SIZE,
    ):
        seen_items = train_seen_by_user.get(user_id)
        candidate_items = build_candidate_items(
            als_items=als_items,
            seen_items=seen_items,
            global_ranked_items=global_ranked_items,
            recent_ranked_items=recent_ranked_items,
            recency_ranked_items=recency_ranked_items,
            popularity_candidates=POPULARITY_CANDIDATES,
        )
        recommended = score_hybrid_candidates(
            candidate_items=candidate_items,
            als_items=als_items,
            als_scores=als_scores,
            config=hybrid_config,
            global_score_arr=global_score_arr,
            recent90_score_arr=recent90_score_arr,
            recency_score_arr=recency_score_arr,
            k=k,
        )
        for rank, article_idx in enumerate(recommended, start=1):
            rows.append(
                {
                    "customer_idx": user_id,
                    "article_idx": int(article_idx),
                    "rank": rank,
                    "model_name": hybrid_config["hybrid_name"],
                    "base_als_model_name": model_name,
                    "als_weight": hybrid_config["als"],
                    "global_weight": hybrid_config["global"],
                    "recent90_weight": hybrid_config["recent90"],
                    "recency_weight": hybrid_config["recency"],
                }
            )

    return pd.DataFrame(rows)


demo_user_ids = (
    sample_submission["customer_idx"]
    .dropna()
    .astype("int32")
    .to_numpy()
)

# Prefer warm users for demo so recommendations are genuinely personalized.
demo_user_ids = [int(u) for u in demo_user_ids if int(u) in train_seen_by_user][:N_DEMO_USERS]

demo_hybrid_recommendations = recommend_hybrid_for_users(
    user_ids=demo_user_ids,
    model=base_als_model,
    model_name=best_als_config["model_name"],
    hybrid_config=best_hybrid_config,
    k=PRIMARY_K,
)

demo_hybrid_recommendations = demo_hybrid_recommendations.merge(
    article_id_map[["article_idx", "article_id"]],
    on="article_idx",
    how="left",
    validate="many_to_one",
)

article_preview_cols = [
    "article_idx",
    "article_id",
    "prod_name",
    "product_type_name",
    "product_group_name",
    "colour_group_name",
    "department_name",
    "section_name",
    "garment_group_name",
]
article_preview_cols = [col for col in article_preview_cols if col in articles.columns]

demo_hybrid_recommendations = demo_hybrid_recommendations.merge(
    articles[article_preview_cols],
    on=["article_idx", "article_id"],
    how="left",
    validate="many_to_one",
)

print_df_info(demo_hybrid_recommendations, "Demo hybrid recommendations")
save_parquet(demo_hybrid_recommendations, HYBRID_ARTIFACTS_DIR / "demo_hybrid_recommendations.parquet")


Demo hybrid recommendations
----------------------------------------------------------------------------------------------------
Shape: (20000, 17)
Memory usage: 13.42 MB


,customer_idx,article_idx,rank,model_name,base_als_model_name,als_weight,global_weight,recent90_weight,recency_weight,article_id,prod_name,product_type_name,product_group_name,colour_group_name,department_name,section_name,garment_group_name
0,0,15988,1,hybrid_als40_global40_recent20,als_f64_reg0p1_it15_alpha20p0,0.4,0.4,0.2,0.0,0568597006,Hayes slim trouser,Trousers,Garment Lower body,Black,Trouser,Womens Tailoring,Trousers
1,0,42626,2,hybrid_als40_global40_recent20,als_f64_reg0p1_it15_alpha20p0,0.4,0.4,0.2,0.0,0673677002,Henry polo. (1),Sweater,Garment Upper body,Black,Knitwear,Womens Tailoring,Knitwear
2,0,67522,3,hybrid_als40_global40_recent20,als_f64_reg0p1_it15_alpha20p0,0.4,0.4,0.2,0.0,0751471001,Pluto RW slacks (1),Trousers,Garment Lower body,Black,Trouser,Womens Everyday Collection,Trousers
3,0,17383,4,hybrid_als40_global40_recent20,als_f64_reg0p1_it15_alpha20p0,0.4,0.4,0.2,0.0,0573716012,Kanta slacks RW,Trousers,Garment Lower body,Black,Trouser,Womens Everyday Collection,Trousers
4,0,6641,5,hybrid_als40_global40_recent20,als_f64_reg0p1_it15_alpha20p0,0.4,0.4,0.2,0.0,0507909001,Rebecca or Delphine shirt,Shirt,Garment Upper body,White,Blouse,Womens Tailoring,Blouses


Saved: /kaggle/working/hm-recommender/artifacts/hybrid_als_popularity/demo_hybrid_recommendations.parquet | 0.21 MB


## 15. Optional Kaggle-style prediction file

In [24]:
def build_kaggle_style_hybrid_predictions(
    users_df: pd.DataFrame,
    model: AlternatingLeastSquares,
    hybrid_config: dict,
    k: int = 12,
) -> pd.DataFrame:
    article_idx_to_article_id = dict(
        zip(article_id_map["article_idx"].astype(int), article_id_map["article_id"].astype(str))
    )

    users_df = users_df.dropna(subset=["customer_idx"]).copy()
    warm_users_df = users_df[users_df["customer_idx"].astype(int).isin(train_seen_by_user.keys())].copy()
    cold_users_df = users_df[~users_df["customer_idx"].astype(int).isin(train_seen_by_user.keys())].copy()

    rows = []

    if len(warm_users_df) > 0:
        warm_user_ids = warm_users_df["customer_idx"].astype("int32").tolist()
        recs_by_user = {}

        for user_id, als_items, als_scores in batch_recommend_als_with_scores(
            model=model,
            user_items_matrix=user_items,
            user_ids=warm_user_ids,
            n_candidates=ALS_CANDIDATES,
            batch_size=RECOMMEND_BATCH_SIZE,
        ):
            seen_items = train_seen_by_user.get(user_id)
            candidate_items = build_candidate_items(
                als_items=als_items,
                seen_items=seen_items,
                global_ranked_items=global_ranked_items,
                recent_ranked_items=recent_ranked_items,
                recency_ranked_items=recency_ranked_items,
                popularity_candidates=POPULARITY_CANDIDATES,
            )
            recs_by_user[user_id] = score_hybrid_candidates(
                candidate_items=candidate_items,
                als_items=als_items,
                als_scores=als_scores,
                config=hybrid_config,
                global_score_arr=global_score_arr,
                recent90_score_arr=recent90_score_arr,
                recency_score_arr=recency_score_arr,
                k=k,
            )

        for row in warm_users_df[["customer_id", "customer_idx"]].itertuples(index=False):
            customer_idx = int(row.customer_idx)
            recs = recs_by_user[customer_idx]
            prediction = " ".join(article_idx_to_article_id[int(item)] for item in recs)
            rows.append({"customer_id": row.customer_id, "prediction": prediction})

    if len(cold_users_df) > 0:
        cold_recs = global_ranked_items[:k].astype(int).tolist()
        cold_prediction = " ".join(article_idx_to_article_id[int(item)] for item in cold_recs)
        for row in cold_users_df[["customer_id", "customer_idx"]].itertuples(index=False):
            rows.append({"customer_id": row.customer_id, "prediction": cold_prediction})

    return pd.DataFrame(rows)


if GENERATE_FULL_SAMPLE_SUBMISSION:
    started = time.time()
    full_prediction_df = build_kaggle_style_hybrid_predictions(
        users_df=sample_submission.dropna(subset=["customer_idx"]),
        model=base_als_model,
        hybrid_config=best_hybrid_config,
        k=12,
    )
    save_csv(full_prediction_df, HYBRID_ARTIFACTS_DIR / "hybrid_submission_like.csv")
    print(f"Generated full hybrid sample-submission-style file in {time.time() - started:.2f} seconds.")
else:
    small_prediction_df = build_kaggle_style_hybrid_predictions(
        users_df=sample_submission.dropna(subset=["customer_idx"]).head(N_DEMO_USERS),
        model=base_als_model,
        hybrid_config=best_hybrid_config,
        k=12,
    )
    display(small_prediction_df.head())
    save_csv(small_prediction_df, HYBRID_ARTIFACTS_DIR / "hybrid_submission_like_demo.csv")
    print("Skipped full submission file. Set GENERATE_FULL_SAMPLE_SUBMISSION = True if needed.")

,customer_id,prediction
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0568597006 0673677002 0751471001 0573716012 05...
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0599580017 0448509014 0688537004 0699080001 07...
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0699080001 0609719001 0458543001 0699081001 05...
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,0720125001 0730683001 0678687001 0738133005 07...
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,0578476001 0586896039 0692721005 0590928013 05...


Saved: /kaggle/working/hm-recommender/artifacts/hybrid_als_popularity/hybrid_submission_like_demo.csv | 0.19 MB
Skipped full submission file. Set GENERATE_FULL_SAMPLE_SUBMISSION = True if needed.


## 16. Final summary

In [25]:
final_summary = {
    "notebook": "06-hybrid-als-popularity.ipynb",
    "purpose": "Blend best tuned ALS with popularity and recency signals.",
    "base_als_config": best_als_config,
    "best_hybrid_config": best_hybrid_config,
    "artifact_dir": str(HYBRID_ARTIFACTS_DIR),
    "decision_summary_file": str(HYBRID_ARTIFACTS_DIR / "hybrid_decision_summary.json"),
    "tuning_leaderboard_file": str(HYBRID_ARTIFACTS_DIR / "hybrid_tuning_leaderboard_k20.csv"),
    "final_leaderboard_file": str(HYBRID_ARTIFACTS_DIR / "final_hybrid_leaderboard_k20.csv"),
    "model_comparison_file": str(HYBRID_ARTIFACTS_DIR / "hybrid_model_comparison.csv"),
    "demo_recommendations_file": str(HYBRID_ARTIFACTS_DIR / "demo_hybrid_recommendations.parquet"),
    "saved_files": sorted([p.name for p in HYBRID_ARTIFACTS_DIR.glob("*")]),
}

print(json.dumps(final_summary, indent=2, default=str))
save_json(final_summary, HYBRID_ARTIFACTS_DIR / "hybrid_final_summary.json")

print()
print("Hybrid ALS + popularity notebook completed successfully.")
print("Upload this executed notebook/output back so we can decide whether to move to time-decayed ALS or finalize hybrid.")

{
  "notebook": "06-hybrid-als-popularity.ipynb",
  "purpose": "Blend best tuned ALS with popularity and recency signals.",
  "base_als_config": {
    "model_name": "als_f64_reg0p1_it15_alpha20p0",
    "stage": "main_grid",
    "factors": 64,
    "alpha": 20.0,
    "regularization": 0.1,
    "iterations": 15
  },
  "best_hybrid_config": {
    "hybrid_name": "hybrid_als40_global40_recent20",
    "als": 0.4,
    "global": 0.4,
    "recent90": 0.2,
    "recency": 0.0
  },
  "artifact_dir": "/kaggle/working/hm-recommender/artifacts/hybrid_als_popularity",
  "decision_summary_file": "/kaggle/working/hm-recommender/artifacts/hybrid_als_popularity/hybrid_decision_summary.json",
  "tuning_leaderboard_file": "/kaggle/working/hm-recommender/artifacts/hybrid_als_popularity/hybrid_tuning_leaderboard_k20.csv",
  "final_leaderboard_file": "/kaggle/working/hm-recommender/artifacts/hybrid_als_popularity/final_hybrid_leaderboard_k20.csv",
  "model_comparison_file": "/kaggle/working/hm-recommender/artif